In [ ]:
import time
import os
from datetime import datetime
from typing import Dict, Any

from bsk_rl.act.actions import ActionBuilder
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

try:
    import pandas as pd
except Exception:
    pd = None

from ray.tune.registry import register_env
from ray.rllib.policy.policy import PolicySpec
from ray.rllib.env.multi_agent_env import MultiAgentEnv
from ray.rllib.algorithms.dqn import DQNConfig, DQN

from Basilisk.architecture import messaging
from collections import Counter  


from bsk_rl import ConstellationTasking
from bsk_rl.sats import ImagingSatellite
from bsk_rl.act import Action, Image
from bsk_rl import obs
from bsk_rl.sim import dyn, fsw
from bsk_rl.scene.targets import UniformTargets, Target
from bsk_rl.data import UniqueImageReward
from bsk_rl.comm import LOSCommunication
from bsk_rl.utils.orbital import walker_delta_args



class TargetAreas(UniformTargets):
    def __init__(self, n_targets: int = 20, priority_distribution=None, radius=6378136.6):
        super().__init__(n_targets, priority_distribution, radius)
        self.lat_min, self.lat_max = -38.0, -25.0
        self.lon_min, self.lon_max = 129.0, 141.0

    def regenerate_targets(self) -> None:
        n_targets = self.n_targets if isinstance(self.n_targets, int) else np.random.randint(
            self.n_targets[0], self.n_targets[1] + 1
        )
        lats = np.random.uniform(self.lat_min, self.lat_max, n_targets)
        lons = np.random.uniform(self.lon_min, self.lon_max, n_targets)
        r = self.radius
        targets = []
        for i, (lat, lon) in enumerate(zip(lats, lons)):
            lat_rad = np.radians(lat)
            lon_rad = np.radians(lon)
            x = r * np.cos(lat_rad) * np.cos(lon_rad)
            y = r * np.cos(lat_rad) * np.sin(lon_rad)
            z = r * np.sin(lat_rad)
            priority = self.priority_distribution() if self.priority_distribution else np.random.uniform(0, 1)
            targets.append(Target(f"SA_Target_{i}", [x, y, z], priority))
        self.targets = targets



class Reallocate(Action):
    def __init__(self, n_sats=4):
        self.action_space = gym.spaces.Discrete(n_sats)
        self.n_actions = self.action_space.n
        self.option = None

    @property
    def builder_type(self):
        return Image.builder_type if isinstance(Image.builder_type, type) else ActionBuilder

    def set_action(self, option: int, **kwargs):
        self.option = option

    def action(self, satellite, state):
        return 0.0



class AdvancedImagingSatellite(ImagingSatellite):
    observation_spec = [
        obs.OpportunityProperties(
            dict(prop="priority"),
            dict(prop="opportunity_open", norm=5700.0),
            n_ahead_observe=5,
        )
    ]
    action_spec = [Image(n_ahead_image=5), Reallocate()]
    dyn_type = dyn.FullFeaturedDynModel
    fsw_type = fsw.SteeringImagerFSWModel


def _tune_access_generation(sat,
                            initial=600.0,    
                            step=300.0,       
                            max_dur=1800.0):  

    candidates = [
        getattr(getattr(sat, "fsw", None), "opportunity_generator", None),
        getattr(getattr(sat, "dynamics", None), "accessGenerator", None),
        getattr(getattr(sat, "dynamics", None), "opportunityGenerator", None),
    ]
    og = next((c for c in candidates if c is not None), None)
    if og is None:
        return

    for name, val in [
        ("initial_generation_duration", float(initial)),
        ("generation_step", float(step)),
        ("max_generation_duration", float(max_dur)),
        ("max_lookahead", float(max_dur)),
    ]:
        if hasattr(og, name):
            setattr(og, name, val)

    # Optimized settings for training speed
    extras = {
        "retask_on_image_complete": False,  
        "max_access_compute_time_s": 1.0,   
    }
    for k, v in extras.items():
        if hasattr(og, k):
            setattr(og, k, v)


class CustomUniqueImageReward(UniqueImageReward):
    
    DEBUG_PROBE = False      
    DEBUG_PROBE_STEPS = 5    

    def __init__(self):
        try:
            super().__init__(data_store_kwargs={"keys": ["imaged", "image", "image_complete"]})
        except TypeError:
            super().__init__()
        self.imaged_by_sat = {f"Sat-{i}": 0 for i in range(4)}
        self._probe_count = 0
        self.imaged_targets = set()

    def _event_key(self, evt):
        for k in ("key", "event", "type", "name"):
            if hasattr(evt, k):
                v = getattr(evt, k)
                if isinstance(v, str):
                    return v
        if isinstance(evt, dict):
            for k in ("key", "event", "type", "name"):
                if k in evt and isinstance(evt[k], str):
                    return evt[k]
        return None

    def _event_sat(self, evt):
        for k in ("agent", "satellite", "sat", "who"):
            if hasattr(evt, k):
                return getattr(evt, k)
        if isinstance(evt, dict):
            for k in ("agent", "satellite", "sat", "who"):
                if k in evt:
                    return evt[k]
        return None

    def _event_target(self, evt):
        for k in ("target", "tgt", "obj"):
            if hasattr(evt, k):
                return getattr(evt, k)
        if isinstance(evt, dict):
            for k in ("target", "tgt", "obj"):
                if k in evt:
                    return evt[k]
        return None

    def _target_id_and_priority(self, tgt):
        if tgt is None:
            return ("<none>", 0.0)
        # object with attributes
        tid = getattr(tgt, "name", None) or getattr(tgt, "id", None)
        pr = getattr(tgt, "priority", None)
        # dict-like
        if tid is None and isinstance(tgt, dict):
            tid = tgt.get("name") or tgt.get("id") or tgt.get("uid")
            pr = tgt.get("priority", pr)
        if tid is None:
            tid = str(tgt)
        try:
            pr = float(pr) if pr is not None else 0.0
        except Exception:
            pr = 0.0
        return (tid, pr)

    def reward(self, new_data_dict):
       
        all_step_targets = []
        for data in new_data_dict.values():
            imgs = getattr(data, "imaged", []) or []
            all_step_targets.extend(imgs)
        occ = Counter(all_step_targets)

        rewards = {}
        for sat_id, data in new_data_dict.items():
            total = 0.0
            new_unique_count = 0
            for tgt in getattr(data, "imaged", []) or []:
                if tgt not in self.data.imaged:
                    prio = float(getattr(tgt, "priority", 0.0))
                    denom = occ.get(tgt, 1) or 1
                    total += self.reward_fn(prio) / denom
                    new_unique_count += 1
            rewards[sat_id] = total
            if new_unique_count > 0:
                self.imaged_by_sat[sat_id] = self.imaged_by_sat.get(sat_id, 0) + new_unique_count

        for sat_id in getattr(self, "imaged_by_sat", {}).keys():
            rewards.setdefault(sat_id, 0.0)

        return rewards

class CustomConstellationTasking(ConstellationTasking, MultiAgentEnv):
    def __init__(self, *args, max_episode_steps=32, **kwargs): 
        super().__init__(*args, **kwargs)
        self.max_episode_steps = int(max_episode_steps)
        self._step_count = 0
        self.remaining_tasks = {"Sat-0": 3, "Sat-1": 3, "Sat-2": 6, "Sat-3": 3}

    def reset(self, *, seed=None, options=None):
        self._step_count = 0
        out = super().reset(seed=seed, options=options) if "seed" in super().reset.__code__.co_varnames else super().reset()
        if isinstance(out, tuple) and len(out) == 2:
            obs, info = out
        else:
            obs, info = out, {}
        self.remaining_tasks = {"Sat-0": 3, "Sat-1": 3, "Sat-2": 6, "Sat-3": 3}
        return obs, info

    def step(self, action_dict: Dict[str, Any]):
        self._step_count += 1
        obs, rews, terms, truncs, infos = super().step(action_dict)

        for agent, r in rews.items():
            if r > 0:
                self.remaining_tasks[agent] = max(0, self.remaining_tasks.get(agent, 0) - 1)

        horizon_reached = self._step_count >= self.max_episode_steps
        no_agents_left = len(self.agents) == 0

        all_keys = set(rews.keys()) | set(terms.keys()) | set(truncs.keys()) | set(self.agents)
        all_done_per_agent = all(terms.get(a, False) or truncs.get(a, False) for a in all_keys) if all_keys else True

        terms["__all__"] = no_agents_left or all_done_per_agent
        truncs["__all__"] = (horizon_reached and not terms["__all__"])

        return obs, rews, terms, truncs, infos


sat_args = {
    "imageAttErrorRequirement": 0.01,
    "imageRateErrorRequirement": 0.01,
    "batteryStorageCapacity": 1e9,
    "storedCharge_Init": 1e9,
    "dataStorageCapacity": 1e12,
    "u_max": 0.4,
    "K1": 0.25,
    "K3": 3.0,
    "omega_max": 0.087,
    "servo_Ki": 5.0,
    "servo_P": 150 / 5,
}
sat_arg_randomizer = walker_delta_args(altitude=800.0, inc=60.0, n_planes=1)

def env_creator(env_config):
    max_episode_steps = env_config.get("max_episode_steps", 32)  # Reduced default
    satellites = [AdvancedImagingSatellite(f"Sat-{i}", sat_args) for i in range(100)]

    for sat in satellites:
        _tune_access_generation(sat, initial=600.0, step=300.0, max_dur=1800.0)  # Optimized settings

    return CustomConstellationTasking(
        satellites=satellites,
        scenario=TargetAreas(n_targets=20),
        rewarder=CustomUniqueImageReward(),
        communicator=LOSCommunication(),
        sat_arg_randomizer=sat_arg_randomizer,
        log_level="WARNING",  # Reduced logging for training
        max_episode_steps=max_episode_steps,
    )

register_env("custom_constellation", env_creator)


print("Initializing environment...")
_sample_env = env_creator({})
obs0, _ = _sample_env.reset()
first_agent = _sample_env.agents[0]
obs_space = _sample_env.observation_space(first_agent)
act_space = _sample_env.action_space(first_agent)
_sample_env.close()
print("Environment initialized successfully!")

config = (
    DQNConfig()
    .api_stack(enable_rl_module_and_learner=False, enable_env_runner_and_connector_v2=False)
    .environment("custom_constellation", env_config={"max_episode_steps": 32})
    .multi_agent(
        policies={"shared_policy": PolicySpec(None, obs_space, act_space)},
        policy_mapping_fn=lambda agent_id, *args, **kwargs: "shared_policy",
    )
    .framework("torch")
    .resources(num_gpus=0)
    .env_runners(
        num_env_runners=1, 
        rollout_fragment_length=8,  
        sample_timeout_s=300.0,     
        num_envs_per_env_runner=1
    )
    .training(
        gamma=0.99,
        lr=5e-4,
        train_batch_size=128,       
        model={"fcnet_hiddens": [128, 128]},  
        n_step=1,                   
        double_q=True,
        dueling=True,
        num_atoms=1,
        noisy=False,
        target_network_update_freq=100,  
        replay_buffer_config={
            "type": "MultiAgentPrioritizedReplayBuffer",
            "capacity": 50000,      
            "alpha": 0.6,
            "beta": 0.4,
        },
    )
)

print("Initializing algorithm...")
algo = DQN(config=config)
print("Algorithm initialized successfully!")

num_iterations = 1000
start_time = time.time()

print(f"Starting training for {num_iterations} iterations...")
print("=" * 60)

try:
    for i in range(num_iterations):
        print(f"\nStarting iteration {i}...")
        iter_start = time.time()
        
        result = algo.train()
        
        iter_time = time.time() - iter_start
        total_time = time.time() - start_time
        
        print(f"Iteration {i} completed in {iter_time:.2f}s (total: {total_time:.2f}s)")
        print(f"Mean reward: {result.get('episode_reward_mean', 'N/A')}")
        print(f"Episodes: {result.get('episodes_total', 'N/A')}")
        print(f"Timesteps: {result.get('timesteps_total', 'N/A')}")
        print("-" * 50)
        
        # Save checkpoint every iteration
        checkpoint_dir = algo.save()
        print(f"Checkpoint saved at {checkpoint_dir}")

except KeyboardInterrupt:
    print("\nTraining interrupted by user")
    # Save final checkpoint on interrupt
    try:
        checkpoint_dir = algo.save()
        print(f"Final checkpoint saved at {checkpoint_dir}")
    except Exception as e:
        print(f"Failed to save final checkpoint: {e}")
        
except Exception as e:
    print(f"\nTraining failed with error: {e}")
    # Try to save anyway
    try:
        checkpoint_dir = algo.save()
        print(f"Emergency checkpoint saved at {checkpoint_dir}")
    except:
        print("Could not save emergency checkpoint")
        
finally:
    print("\nTraining session completed!")
    print(f"Total training time: {time.time() - start_time:.2f} seconds")

# Test the trained policy
print("\nTesting trained policy...")
test_env = env_creator({"max_episode_steps": 32})
obs, info = test_env.reset()
terminated = {"__all__": False}
truncated = {"__all__": False}
total_reward = 0
steps = 0

while not terminated["__all__"] and not truncated["__all__"] and steps < 50:
    actions = {}
    for agent_id in test_env.agents:
        # Use the trained policy to select actions
        actions[agent_id] = algo.compute_single_action(obs[agent_id], policy_id="shared_policy")
    
    obs, rewards, terminated, truncated, info = test_env.step(actions)
    total_reward += sum(rewards.values())
    steps += 1

print(f"Test completed: {steps} steps, total reward: {total_reward}")
test_env.close()

print("\nTraining completed successfully!")

In [ ]:
print("\n=== TESTING PHASE (With Fault Injection at Step 8) ===")

# Load the saved policy
algo = DQN(config=config)
algo.restore(checkpoint_dir)

def test_env_creator(env_config):
    max_episode_steps = env_config.get("max_episode_steps", 64)
    satellites = [AdvancedImagingSatellite(f"Sat-{i}", sat_args) for i in range(4)]

    for sat in satellites:
        _tune_access_generation(sat, initial=1800.0, step=120.0, max_dur=3600.0)

    return CustomConstellationTasking(
        satellites=satellites,
        scenario=TargetAreas(n_targets=12),  
        rewarder=CustomUniqueImageReward(),
        communicator=LOSCommunication(),
        sat_arg_randomizer=sat_arg_randomizer,
        log_level="INFO",
        max_episode_steps=max_episode_steps,
    )

test_env = test_env_creator({})

def inject_failure(sat_index: int):
    """Force a satellite 'dead' by zeroing power each step."""
    sat = test_env.unwrapped.satellites[sat_index]
    def isnt_alive(log_failure=False, _sat=sat):
        death_message = messaging.PowerStorageStatusMsgPayload()
        death_message.storageLevel = 0.0
        _sat.dynamics.powerMonitor.batPowerOutMsg.write(death_message)
        return _sat.dynamics.is_alive(log_failure=log_failure) and _sat.fsw.is_alive(log_failure=log_failure)
    sat.is_alive = isnt_alive

def find_nearest_satellite(faulty_sat_index):
    """Find the nearest operational satellite to transfer tasks to."""
    operational_sats = []
    for i, sat in enumerate(test_env.unwrapped.satellites):
        if i != faulty_sat_index and sat.name in test_env.agents:
            operational_sats.append(i)
    
    if operational_sats:
        return operational_sats[0]  
    return None

max_steps = 45
max_wallclock_s = 180

observations, _ = test_env.reset()
test_env.remaining_tasks = {"Sat-0": 3, "Sat-1": 3, "Sat-2": 3, "Sat-3": 3}

current_step = 0
episode_reward = 0.0
t0 = time.time()

print(f"Initial state (Step {current_step}):")
for sat_name in ["Sat-0", "Sat-1", "Sat-2", "Sat-3"]:
    imaged = getattr(test_env.rewarder, "imaged_by_sat", {}).get(sat_name, 0)
    remaining = test_env.remaining_tasks.get(sat_name, 0)
    print(f"{sat_name}: imaged = {imaged} tasks = {imaged + remaining} remaining = {remaining}")

while current_step < max_steps and test_env.agents:
    if time.time() - t0 > max_wallclock_s:
        print(f"[Test] Wallclock timeout ({max_wallclock_s}s). Breaking.")
        break

    if current_step == 8 and "Sat-1" in test_env.agents:
        print(f"\n=== INJECTING FAULT AT STEP {current_step} ===")
        print("Before fault injection:")
        for sat_name in ["Sat-0", "Sat-1", "Sat-2", "Sat-3"]:
            imaged = getattr(test_env.rewarder, "imaged_by_sat", {}).get(sat_name, 0)
            remaining = test_env.remaining_tasks.get(sat_name, 0)
            print(f"{sat_name}: imaged = {imaged} tasks = {imaged + remaining} remaining = {remaining}")
        
        nearest_sat_index = find_nearest_satellite(1)  
        if nearest_sat_index is not None:
            nearest_sat_name = f"Sat-{nearest_sat_index}"
            sat1_tasks = test_env.remaining_tasks.get("Sat-1", 0)
            test_env.remaining_tasks[nearest_sat_name] += sat1_tasks
            test_env.remaining_tasks["Sat-1"] = 0
            
            print(f"\nTransferring {sat1_tasks} tasks from Sat-1 to {nearest_sat_name}")
            print("After fault injection and task transfer:")
            for sat_name in ["Sat-0", "Sat-1", "Sat-2", "Sat-3"]:
                imaged = getattr(test_env.rewarder, "imaged_by_sat", {}).get(sat_name, 0)
                remaining = test_env.remaining_tasks.get(sat_name, 0)
                print(f"{sat_name}: imaged = {imaged} tasks = {imaged + remaining} remaining = {remaining}")
        
        inject_failure(1)
        print(f"\n*** FAULT INJECTED: Sat-1 is now faulty ***")
        print("=================================\n")

    actions = {}
    for agent in test_env.agents:
        pid = algo.config.policy_mapping_fn(agent)
        action = algo.compute_single_action(observations[agent], policy_id=pid)
        actions[agent] = action

    try:
        observations, rewards, terminations, truncations, infos = test_env.step(actions)
        episode_reward += sum(rewards.values())
    except KeyError as e:
        print(f"Warning: KeyError encountered at step {current_step}: {e}")
        print("This is likely due to the fault injection. Continuing simulation...")
        if "Sat-1" in observations:
            del observations["Sat-1"]
        rewards = {agent: 0.0 for agent in test_env.agents}

    current_step += 1



print(f"\nTest episode reward: {episode_reward}")